# ARIA - LITE

ARIA Lite is a lightweight GraphRAG-based biomedical research assistant focused on breast cancer AI literature.

The project combines two retrieval paradigms:

1. Semantic Retrieval
   Dense vector embeddings are used to retrieve papers semantically related to a user query.

2. Graph-Based Retrieval
   Biomedical entities such as genes and drugs are extracted from papers and represented as relationships in a lightweight knowledge graph.

By combining these two approaches, the system aims to provide more grounded and explainable retrieval compared to traditional vector-only RAG systems.

The project is intentionally scoped for rapid iteration and learning:
- ~300-500 PubMed papers
- Abstract-only corpus
- Lightweight graph construction
- Citation-grounded responses

Core technologies:
- PubMed / Entrez API
- SciSpacy
- Sentence Transformers
- FAISS
- Python + Google Colab

End Goal:
Build a small but functional biomedical GraphRAG system capable of retrieving relevant breast cancer AI papers and generating grounded answers with PMID citations.

# 3A_entity_extraction.ipynb

Purpose: This notebook performs biomedical entity extraction on the cleaned PubMed dataset generated in Week 2.

The goal is to convert unstructured biomedical text into structured, graph-ready entities that can later be used for building a knowledge graph and enabling hybrid GraphRAG-style retrieval.

Specifically, this notebook:

- Extracts biomedical entities from each paper using SciSpacy (genes, proteins, diseases, and chemical compounds)
- Normalizes entity text to ensure consistency across papers (e.g., merging variations like "BRCA1", "brca1")
- Filters out noisy or non-informative entities using frequency-based and blacklist-based filtering
- Aggregates entities at the document (PMID) level to build a structured mapping:
  - Paper → Entities
- Produces a cleaned entity dataset that will serve as the foundation for:
  - Knowledge graph construction (Week 4)
  - Hybrid retrieval (graph + vector search)
  - Downstream LLM grounding and explanation

Why this step is important?

While embeddings capture semantic similarity, they do not explicitly model biomedical structure.

This step introduces a symbolic layer by identifying medically relevant concepts, enabling:

- Better interpretability of retrieved results
- Structured relationships between papers and biological entities
- Improved grounding for LLM responses in later stages

Output of this notebook:
- `entities.json`: a mapping of each PubMed ID to its associated filtered biomedical entities

This structured entity layer will be used directly for building the ARIA-Lite knowledge graph in the next step

In [1]:
# ============================================================
# SECTION 1 — Install Libraries
# ============================================================

!pip install spacy scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz (119.8 MB)
  Preparing metadata (setup.py) ... done
  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz (119.8 MB)
  Preparing metadata (setup.py) ... done


In [2]:
# ============================================================
# SECTION 2 — Imports
# ============================================================

from google.colab import drive
import os

import json
import spacy
from collections import Counter, defaultdict
import re

In [3]:
# ============================================================
# SECTION 3 — Mount Google Drive
# ============================================================

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# ============================================================
# SECTION 4 — Project Paths and data loading
# ============================================================

PROJECT_ROOT = "/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2"

INPUT_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "structured_papers.json")

OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "clean_papers_with_entities.json")

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Output path:", OUTPUT_PATH)

with open(INPUT_PATH, "r") as f:
    papers = json.load(f)

print(f"Loaded {len(papers)} papers")

Project root: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2
Input path: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/structured_papers.json
Output path: /content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/clean_papers_with_entities.json
Loaded 475 papers


In [5]:
# ============================================================
# SECTION 5 — Load SciSpacy Models
# ============================================================

print("Loading SciSpacy models...")

nlp_bc5cdr = spacy.load("en_ner_bc5cdr_md")

nlp_bionlp13cg = spacy.load("en_ner_bionlp13cg_md")

nlp_models = {
    "bc5cdr": nlp_bc5cdr,
    "bionlp": nlp_bionlp13cg
}

print("Models loaded")

Loading SciSpacy models...


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


Models loaded


In [6]:
# ============================================================
# SECTION 6A — Biomedical NER
# ============================================================

ALLOWED_LABELS = {
    "DISEASE",
    "CHEMICAL",
    "CANCER",
    "CELL",
    "GENE_OR_GENE_PRODUCT",
    "ORGANISM",
    "TISSUE"
}

def extract_entities_multi_model(
    text,
    nlp_models
):
    entities = []

    for model_name, nlp in nlp_models.items():

        doc = nlp(text)

        for ent in doc.ents:

            label = ent.label_

            if label in ALLOWED_LABELS:

                entities.append({
                    "text": ent.text,
                    "label": label,
                    "source_model": model_name
                })

    return entities

In [7]:
# ============================================================
# SECTION 6B — ML / CS Entity Extraction
# ============================================================

ML_CS_ENTITIES = [

    'AI',
    'artificial intelligence',
    'AUC',
    'AUPRC',
    'AUROC',
    'Autoencoder',
    'Bayesian logistic regression',
    'Bayesian modelling',
    'benchmarking',
    'biomarker discovery',
    'calibration curves',
    'catastrophic forgetting',
    'CIBERSORT',
    'class activation mapping',
    'classification',
    'CNNs',
    'computer vision',
    'confusion matrices',
    'convolutional neural networks',
    'Cox regression',
    'cross-validation',
    'data pre-processing',
    'decision curve analysis',
    'deep learning',
    'diagnostic accuracy',
    'digital pathology',
    'ensemble feature selection',
    'ensemble methods',
    'ESTIMATE',
    'experience replay',
    'explainable artificial intelligence',
    'extended-connectivity fingerprint',
    'external validation',
    'feature selection',
    'foundation models',
    'generalized linear mixed models',
    'generative artificial intelligence',
    'hyperparameter tuning',
    'interpretability',
    'Lasso regression',
    'logistic regression',
    'machine learning',
    'Mendelian randomization',
    'meta-analysis',
    'ML',
    'molecular docking',
    'multi-omics',
    'multilayer perceptrons',
    'multitask',
    'neural network',
    'nomograms',
    'omics',
    'pipeline',
    'precision oncology',
    'predictive models',
    'PROBAST+AI',
    'prognostic model',
    'Random Forest',
    'random forest',
    'random search',
    'recursive feature elimination',
    'ResNet-50',
    'RNA-seq',
    'ROC',
    'scRNA-seq',
    'SHAP',
    'SHapley Additive exPlanations',
    'single-cell RNA-seq',
    'spatial transcriptomics',
    'ssGSEA',
    'support vector machine',
    'systematic review',
    't-distributed stochastic neighbor embedding',
    'TabPFN',
    'transcriptomics',
    'transfer learning',
    'validation',
    'WGNA',
    'XAI',
    'XGBoost'
]

# ------------------------------------------------------------
# Compile regex patterns
# ------------------------------------------------------------

ML_CS_PATTERNS = [

    (
        term,
        re.compile(
            rf"\b{re.escape(term)}\b",
            re.IGNORECASE
        )
    )

    for term in ML_CS_ENTITIES
]

def extract_ml_cs_entities(
    text,
    source_model="ml_cs_rule_based"
):

    entities = []

    for term, pattern in ML_CS_PATTERNS:

        if pattern.search(text):

            entities.append({
                "text": term,
                "label": "ML_METHOD",
                "source_model": source_model
            })

    return entities

In [8]:
# ============================================================
# SECTION 7 — Normalization
# ============================================================

def normalize(text):

    text = text.lower()
    text = re.sub(r"[^a-z0-9\- ]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [9]:
# ============================================================
# SECTION 8 — Run Extraction
# ============================================================

section_entities = defaultdict(list)

all_entities = []

for paper in papers:

    pmid = paper["pmid"]

    sections = paper.get(
        "sections",
        []
    )

    # --------------------------------------------------------
    # Process each section independently
    # --------------------------------------------------------

    for section in sections:

        section_id = section["section_id"]

        text = section["text"]

        # ----------------------------------------------------
        # Biomedical entities
        # ----------------------------------------------------

        med_ents = extract_entities_multi_model(
            text,
            nlp_models
        )

        # ----------------------------------------------------
        # ML / CS entities
        # ----------------------------------------------------

        cs_entities = extract_ml_cs_entities(
            text
        )

        ents = med_ents + cs_entities

        # ----------------------------------------------------
        # Store entities
        # ----------------------------------------------------

        for e in ents:

            normalized_entity = normalize(
                e["text"]
            )

            entity_record = {

                "entity": normalized_entity,

                "label": e["label"]
            }

            section_entities[
                section_id
            ].append(entity_record)

            all_entities.append(
                (
                    normalized_entity,
                    e["label"]
                )
            )

print("\nExtraction complete")
print("Total entities:", len(all_entities))


Extraction complete
Total entities: 12543


In [10]:
# ============================================================
# SECTION 9 — Aggregate Entity Frequencies
# ============================================================

entity_counter = Counter()

for entity, label in all_entities:
    key = (entity, label)
    entity_counter[key] += 1

print("\nUnique entities:", len(entity_counter))


Unique entities: 3155


In [11]:
# ============================================================
# SECTION 10 — Frequency Filtering
# ============================================================

MIN_FREQ = 2

filtered_entities = {
    k: v
    for k, v in entity_counter.items()
    if v >= MIN_FREQ
}

print(
    "Entities after filtering:",
    len(filtered_entities)
)

Entities after filtering: 1112


In [12]:
# ============================================================
# SECTION 11 — Build Final Filtered Mapping
# ============================================================

filtered_section_entities = defaultdict(list)

for section_id, ents in section_entities.items():

    seen_entities = set()

    for e in ents:

        key = (e["entity"], e["label"])

        # ----------------------------------------------------
        # Frequency filtering
        # ----------------------------------------------------

        if key not in filtered_entities:
            continue

        # ----------------------------------------------------
        # Deduplication
        # ----------------------------------------------------

        if key in seen_entities:
            continue

        seen_entities.add(key)

        filtered_section_entities[section_id].append({"entity": e["entity"], "label": e["label"]})

In [13]:

# ============================================================
# SECTION 12 — Dataset Statistics
# ============================================================

total_sections = len(filtered_section_entities)

total_filtered_entities = sum(len(v) for v in filtered_section_entities.values())

print("\n===================================")
print("ENTITY EXTRACTION STATS")
print("===================================")

print("Sections with entities:",
      total_sections)

print("Total filtered entities:",
      total_filtered_entities)


ENTITY EXTRACTION STATS
Sections with entities: 1079
Total filtered entities: 7772


In [14]:
# ============================================================
# SECTION 13 — Example Inspection
# ============================================================

sample_key = list(
    filtered_section_entities.keys()
)[0]

print("\n===================================")
print("SAMPLE OUTPUT")
print("===================================")

print("Section ID:", sample_key)

print("\nEntities:")

for item in filtered_section_entities[
    sample_key
][:10]:

    print(item)


SAMPLE OUTPUT
Section ID: 42104431_BACKGROUND

Entities:
{'entity': 'breast cancer', 'label': 'DISEASE'}
{'entity': 'death', 'label': 'DISEASE'}
{'entity': 'tumor', 'label': 'DISEASE'}
{'entity': 'alns', 'label': 'DISEASE'}
{'entity': 'aln', 'label': 'CHEMICAL'}
{'entity': 'breast cancer', 'label': 'CANCER'}
{'entity': 'women', 'label': 'ORGANISM'}
{'entity': 'tumor cells', 'label': 'CELL'}
{'entity': 'alns', 'label': 'CANCER'}
{'entity': 'patients', 'label': 'ORGANISM'}


In [15]:
# ============================================================
# SECTION 14 — Save Output
# ============================================================

with open(OUTPUT_PATH, "w") as f:

    json.dump(
        filtered_section_entities,
        f,
        indent=2
    )

print("\n===================================")
print("SAVE COMPLETE")
print("===================================")

print("Saved entities to:")
print(OUTPUT_PATH)


SAVE COMPLETE
Saved entities to:
/content/drive/MyDrive/Colab_Notebooks/LLMs/ARIA_Lite_v2/data/processed/clean_papers_with_entities.json
